# Getting Started with MLflow  
## Math 103.1: Predictive Analytics for Text

### Purpose of this Notebook

This notebook introduces MLflow as a practical tool for tracking machine learning experiments.

By the end of this notebook, you should be able to:

1. explain what MLflow tracks;
2. distinguish experiments, runs, parameters, metrics, and artifacts;
3. log a simple MLflow run;
4. connect MLflow tracking to reproducibility and ethical alignment.

MLflow is useful because model scores are not enough. We also need to know how those scores were produced.

## 1. Why MLflow Matters in This Course

In text analytics, we often try many combinations:

- different preprocessing choices;
- different vectorizers;
- different models;
- different hyperparameters;
- different evaluation metrics.

Without tracking, it becomes difficult to remember which experiment produced which result.

MLflow helps us keep an organized record of machine learning experiments.

## 2. What MLflow Tracks

MLflow can track:

1. parameters;
2. metrics;
3. artifacts;
4. models;
5. notes about the experiment.

In Math 103.1, we will use MLflow mainly to make model experiments traceable.

## 3. Basic MLflow Vocabulary

### Experiment

An experiment is a collection of related runs.

Example:

```text
week04_text_classification
```

### Run

A run is one specific trial.

Example:

```text
logreg_tfidf_baseline
```

### Parameter

A parameter is an input setting.

Example:

```text
max_features = 5000
```

### Metric

A metric is an evaluation result.

Example:

```text
macro_f1 = 0.59
```

### Artifact

An artifact is an output file.

Example:

```text
classification_report.txt
```

## 4. Check Whether MLflow Is Installed

Run the next cell.

If MLflow is not installed, your instructor may provide an environment file or installation instructions.

In [1]:
try:
    import mlflow
    print("MLflow is installed.")
    print("MLflow version:", mlflow.__version__)
except ImportError:
    print("MLflow is not installed in this environment.")
    print("Ask your instructor for the course environment setup instructions.")

MLflow is installed.
MLflow version: 3.14.0


## 5. Minimal MLflow Example

The next cell logs one simple run.

It records:

- model name;
- vectorizer name;
- random seed;
- accuracy;
- macro F1-score.

These values are only demonstration values.

In [2]:
from pathlib import Path

import mlflow

# Create a local folder for MLflow tracking files

tracking_dir = Path("mlflow_tracking")

tracking_dir.mkdir(exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{tracking_dir / 'mlflow.db'}")

print("MLflow tracking URI:")

print(mlflow.get_tracking_uri())

# mlflow.set_experiment("week01_mlflow_demo")
mlflow.set_experiment("week00_mlflow_demo")

with mlflow.start_run(run_name="logistic_regression_baseline_demo"):

    mlflow.log_param("model", "logistic_regression")
    mlflow.log_param("vectorizer", "tfidf")
    mlflow.log_param("random_seed", 103)

    mlflow.log_metric("accuracy", 0.62)
    mlflow.log_metric("macro_f1", 0.59)

print("MLflow demo run completed.")

MLflow tracking URI:
sqlite:///mlflow_tracking/mlflow.db
MLflow demo run completed.


## 6. Why Parameters and Metrics Must Be Logged Together

A metric without context is incomplete.

Poor record:

```text
Accuracy = 0.62
```

Better record:

```text
Logistic Regression with TF-IDF, random_seed=103, achieved validation accuracy = 0.62 and macro F1 = 0.59.
```

MLflow helps you create the better kind of record.

## 7. A Small Classification Example

The next example uses a tiny artificial dataset.

This is not a serious model. It is only meant to show the MLflow workflow.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

texts = [
    "excellent movie with strong acting",
    "bad movie and weak story",
    "great film and excellent scenes",
    "poor acting and boring plot",
    "wonderful story and great ending",
    "terrible film with bad acting",
    "excellent and enjoyable",
    "boring and weak"
]

labels = [1, 0, 1, 0, 1, 0, 1, 0]

X_train, X_val, y_train, y_val = train_test_split(
    texts,
    labels,
    test_size=0.25,
    random_state=103,
    stratify=labels
)

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

model = LogisticRegression(random_state=103)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_val_tfidf)

accuracy = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average="macro")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)

Accuracy: 1.0
Macro F1: 1.0


## 8. Log the Small Classification Example

The next cell logs the toy classification experiment in MLflow.

In [4]:
import mlflow

mlflow.set_experiment("week00_toy_text_classification")

with mlflow.start_run(run_name="logreg_tfidf_toy_dataset"):

    mlflow.log_param("task", "binary_classification")
    mlflow.log_param("dataset", "toy_sentiment_dataset")
    mlflow.log_param("text_column", "texts")
    mlflow.log_param("target_variable", "labels")
    mlflow.log_param("split_seed", 103)
    mlflow.log_param("vectorizer", "TfidfVectorizer")
    mlflow.log_param("model", "LogisticRegression")

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("macro_f1", macro_f1)

print("Toy classification run logged.")

Toy classification run logged.


## 9. Logging an Artifact

An artifact is a file saved from an experiment.

The next cell creates a short text file and logs it as an artifact.

In [5]:
from pathlib import Path
import mlflow

artifact_dir = Path("mlflow_demo_outputs")
artifact_dir.mkdir(exist_ok=True)

report_path = artifact_dir / "toy_classification_report.txt"

report_text = f'''Toy Classification Report

Model: Logistic Regression
Vectorizer: TF-IDF
Accuracy: {accuracy}
Macro F1: {macro_f1}

Important note:
This is a toy example. The result should not be overinterpreted.
'''

report_path.write_text(report_text, encoding="utf-8")

mlflow.set_experiment("week00_toy_text_classification")

with mlflow.start_run(run_name="log_artifact_demo"):
    mlflow.log_param("artifact_type", "text_report")
    mlflow.log_artifact(str(report_path))

print(f"Logged artifact: {report_path}")

Logged artifact: mlflow_demo_outputs/toy_classification_report.txt


## 10. Opening the MLflow User Interface

After running MLflow code, you can inspect the results using the MLflow UI.

In the terminal, go to your project folder and run:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow_tracking/mlflow.db
```

Then open the local address shown in the terminal, commonly:

```text
http://127.0.0.1:5000
```

The MLflow UI allows you to compare runs in a table.

## 11. Recommended MLflow Logging in Math 103.1

For each model experiment, log at least the following.

### Parameters

```text
task
dataset
target_variable
text_column
split_seed
vectorizer
model
important_model_settings
```

### Metrics for Classification

```text
accuracy
macro_f1
precision
recall
```

### Metrics for Regression

```text
mae
rmse
r2
```

### Artifacts

```text
classification_report.txt
confusion_matrix.png
results_table.csv
model_summary.txt
```

## 12. MLflow and Reproducibility

A result is more useful when another person can understand how it was produced.

For reproducibility, record:

- the dataset used;
- the target variable;
- the train-validation-test split;
- the random seed;
- the vectorizer;
- the model;
- the hyperparameters;
- the evaluation metrics;
- important limitations.

MLflow does not replace understanding. It helps document the experiment so that understanding is easier.

## 13. MLflow and Ethical Alignment

In this course, model results should not be treated as magic numbers.

MLflow supports responsible work because it helps answer:

- Was the result produced by an actual experiment?
- Were the parameters documented?
- Were the metrics reported honestly?
- Can the result be checked?
- Were alternative models compared fairly?
- Were limitations recorded?

This connects to the course principle that text prediction is an experiment, not a one-click modeling exercise.

## 14. Common MLflow Mistakes

### Mistake 1: Logging only metrics

Poor:

```python
mlflow.log_metric("accuracy", accuracy)
```

Better:

```python
mlflow.log_param("model", "LogisticRegression")
mlflow.log_param("vectorizer", "TF-IDF")
mlflow.log_param("random_seed", 103)
mlflow.log_metric("accuracy", accuracy)
mlflow.log_metric("macro_f1", macro_f1)
```

### Mistake 2: Forgetting the random seed

If you do not record the random seed, the experiment may be harder to reproduce.

### Mistake 3: Comparing models unfairly

Do not compare models using different data splits unless you clearly explain the difference.

### Mistake 4: Treating the highest accuracy as automatically best

For imbalanced classification tasks, accuracy may be misleading. Macro F1-score may be more informative.

## 15. Practice Reflection

Edit the cell below.

Answer in 3 to 5 sentences:

**Why is MLflow useful for making text prediction experiments traceable?**

### My Reflection

Write your answer here.

## 16. Minimal MLflow Checklist

Before submitting an MLflow-based activity, check the following:

- The experiment has a meaningful name.
- Each run has a meaningful name.
- Important parameters are logged.
- Important metrics are logged.
- The dataset and target variable are identified.
- The random seed or split information is recorded.
- Important artifacts are saved when required.
- The best result is interpreted, not merely reported.
- Limitations are stated honestly.

## 17. Final Reminder

MLflow is not required because experiments are complicated.

MLflow is required because experiments must be traceable.

In Math 103.1, a good model result should answer not only “What score did you get?” but also “How did you get that score, and can someone check it?”